# 2) Download and save GP (last 730 days)


In [ ]:
# Inputs: none | Process: import libs | Outputs: env ready
import requests
import pandas as pd
from io import StringIO
import time
from pathlib import Path
from datetime import datetime


In [ ]:
# Inputs: env vars or hardcoded creds | Process: set config | Outputs: constants
USERNAME = "aaeushsingh98@gmail.com"
PASSWORD = "RA5wtMpC67!!6r6AB12"
BATCH_SIZE = 100000
THROTTLE_SECONDS = 1
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
for d in [RAW_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
# Inputs: creds + max-id | Process: fetch GP by NORAD ranges (EPOCH>now-730) | Outputs: gp_df
with requests.Session() as s:
    r = s.post("https://www.space-track.org/ajaxauth/login", data={"identity": USERNAME, "password": PASSWORD})
    if r.status_code != 200 or "Failed" in r.text:
        raise RuntimeError("Login failed")

    max_id_url = (
        "https://www.space-track.org/basicspacedata/query/class/satcat/"
        "orderby/NORAD_CAT_ID%20desc/limit/1/format/csv"
    )
    r = s.get(max_id_url)
    r.raise_for_status()
    max_norad = int(pd.read_csv(StringIO(r.text))["NORAD_CAT_ID"].iloc[0])

    gp_base = "https://www.space-track.org/basicspacedata/query/class/gp"
    batches = []
    for start_id in range(1, max_norad + 1, BATCH_SIZE):
        end_id = min(start_id + BATCH_SIZE - 1, max_norad)
        url = f"{gp_base}/NORAD_CAT_ID/{start_id}--{end_id}/EPOCH/%3Enow-730/orderby/NORAD_CAT_ID,EPOCH/format/csv"
        t0 = time.time()
        rr = s.get(url)
        if rr.status_code == 429:
            wait_s = int(rr.headers.get("Retry-After", THROTTLE_SECONDS))
            time.sleep(wait_s)
            continue
        rr.raise_for_status()
        batches.append(pd.read_csv(StringIO(rr.text)))
        time.sleep(max(THROTTLE_SECONDS, 1 - (time.time()-t0)))

gp_df = pd.concat(batches, ignore_index=True) if batches else pd.DataFrame()


In [ ]:
# Inputs: gp_df | Process: save raw+latest | Outputs: CSV paths
TS = datetime.now().strftime('%Y%m%d_%H%M%S')
raw_path = RAW_DIR / f"gp_{TS}.csv"
latest_path = PROCESSED_DIR / f"gp_{TS}.csv"  # keep timestamped in processed too
if not gp_df.empty:
    gp_df.to_csv(raw_path, index=False)
    gp_df.to_csv(latest_path, index=False)
print("Saved:", raw_path if 'raw_path' in locals() else 'n/a', "and", latest_path if 'latest_path' in locals() else 'n/a')
